# env_sensor_hub — I2C Environmental Sensor Hub

STM32L031K6T6 MCU on 3.3V rail (MIC5219 LDO from 5V header). Two I2C sensors: SHT31-DIS-B (temp/humidity) and BMP388 (barometric pressure). Shared I2C bus with 4.7k pull-ups. UART debug header.

In [ ]:
import pathlib
import hw_toolkit as hw

board = hw.Board("env_sensor_hub")
print(f"Board created: {board.project_id}")

## MCU — STM32L031K6T6 (Cortex-M0+, low-power)

In [ ]:
# STM32L031K6T6: ultra-low-power MCU, 32KB flash, LQFP-32
mcu = board.module(
    id="mcu",
    category="mcu",
    mpn="STM32L031K6T6",
    package="LQFP-32",
    price_usd=2.10,
    manufacturer="STMicroelectronics",
)
print(mcu)

## LDO Regulator — MIC5219-3.3YM5-TR (5V → 3.3V, modeled as board.module with VIN/VOUT/GND pins)

In [ ]:
# MIC5219-3.3YM5-TR: 500mA LDO, 5V input, 3.3V output
# Pins synthesized: VIN, VOUT, GND, EN
ldo = board.module(
    id="ldo",
    category="ldo_regulator",
    mpn="MIC5219-3.3YM5-TR",
    package="SOT-23-5",
    price_usd=0.65,
    manufacturer="Microchip",
)
print(ldo)

## 5V Input Header

In [ ]:
# 2-pin 5V power input header (pin 1 = VIN, pin 2 = GND)
hdr5v = board.module(
    id="hdr5v",
    category="connector",
    mpn="PinHeader_2.54mm_2pin",
    package="THT",
    price_usd=0.10,
    manufacturer="Generic",
)
print(hdr5v)

## Sensor 1 — SHT31-DIS-B (Temperature + Humidity)

In [ ]:
# SHT31-DIS-B: high-accuracy temp/humidity, I2C 0x44/0x45, DFN-8
sht = board.module(
    id="sht",
    category="environmental_sensor",
    mpn="SHT31-DIS-B",
    package="DFN-8",
    price_usd=4.20,
    manufacturer="Sensirion",
)
print(sht)

## Sensor 2 — BMP388 (Barometric Pressure)

In [ ]:
# BMP388: barometric pressure + temperature, I2C 0x76/0x77, LGA-10
bmp = board.module(
    id="bmp",
    category="pressure_sensor",
    mpn="BMP388",
    package="LGA-10",
    price_usd=3.50,
    manufacturer="Bosch",
)
print(bmp)

## UART Debug Header

In [ ]:
# 3-pin debug UART header (TX, RX, GND)
dbghdr = board.module(
    id="dbghdr",
    category="connector",
    mpn="PinHeader_2.54mm_3pin",
    package="THT",
    price_usd=0.10,
    manufacturer="Generic",
)
print(dbghdr)

## Passives — Decoupling Caps + I2C Pull-ups

In [ ]:
# Decoupling caps: one per IC rail
c_mcu  = board.capacitor("C_MCU",  "100nF", package="0402")
c_ldo  = board.capacitor("C_LDO",  "100nF", package="0402")
c_sht  = board.capacitor("C_SHT",  "100nF", package="0402")
c_bmp  = board.capacitor("C_BMP",  "100nF", package="0402")

# I2C pull-up resistors: 4.7k on SDA and SCL to 3.3V
r_sda  = board.resistor("R_SDA",  "4k7",  package="0402")
r_scl  = board.resistor("R_SCL",  "4k7",  package="0402")

print("Passives:", c_mcu.id, c_ldo.id, c_sht.id, c_bmp.id, r_sda.id, r_scl.id)

## Power Nets

In [ ]:
# 5V rail: header VIN → LDO VIN
v5   = board.power("v5",   voltage_v=5.0)
v5  += "hdr5v.VIN", "ldo.VIN"

# 3.3V rail: LDO VOUT → MCU + sensors + pull-up resistors + decoupling
v3v3 = board.power("v3v3", voltage_v=3.3)
v3v3 += "ldo.VOUT", "mcu.VDD", "sht.VDD", "bmp.VDD"
v3v3 += "r_sda.A", "r_scl.A"
v3v3 += "c_mcu.POS", "c_ldo.POS", "c_sht.POS", "c_bmp.POS"

# GND: everything
gnd  = board.gnd()
gnd += "hdr5v.GND", "ldo.GND", "mcu.GND", "sht.GND", "bmp.GND"
gnd += "dbghdr.GND"
gnd += "c_mcu.NEG", "c_ldo.NEG", "c_sht.NEG", "c_bmp.NEG"

# LDO EN: tie to VIN for always-on
ldo_en = board.signal("ldo_en", protocol="gpio")
ldo_en += "ldo.EN", "hdr5v.VIN"

print("Power nets OK")

## I2C Bus — env (shared by SHT31 + BMP388)

In [ ]:
# Single I2C bus: MCU master, SHT31 + BMP388 as devices
sda, scl = board.i2c("env")

# SDA: MCU → SHT → BMP + pull-up resistor other leg
sda += "mcu.SDA", "sht.SDA", "bmp.SDI"
sda += "r_sda.B"

# SCL: MCU → SHT → BMP + pull-up resistor other leg
scl += "mcu.SCL", "sht.SCL", "bmp.SCK"
scl += "r_scl.B"

print("SDA members:", sda.members)
print("SCL members:", scl.members)

## UART Debug Bus

In [ ]:
# UART debug: MCU USART1 TX/RX to debug header
tx, rx = board.uart("dbg")
tx += "mcu.TX", "dbghdr.TX"
rx += "mcu.RX", "dbghdr.RX"

print("UART TX:", tx.members)
print("UART RX:", rx.members)

## Board Summary

In [ ]:
print(board.summary())

## ERC Check

In [ ]:
board.check_erc(expected_codes=(
    "pin_not_connected",          # intentional NCs on synthesized symbols
    "lib_symbol_issues",          # hwagent lib synthesized at runtime
    "pin_to_pin",                 # LDO EN tied to VIN for always-on
    "power_pin_not_driven",       # connector power pins without PWR_FLAG
    "unconnected_wire_endpoint",  # synthesized wire-layout artifact
    "footprint_link_issues",      # synthesized footprint names don't resolve to KiCad stock lib
))
print("ERC passed")

## Export KiCad Project

In [ ]:
import pathlib

out = pathlib.Path("/Users/juanantonioluera/ws/hw-toolkit/docs/projects/env_sensor_hub/env_sensor_hub.zip")
board.export_kicad(out, unzip=True)

print(f"Exported: {out}")
print(f"Zip exists: {out.exists()}")